<a href="https://colab.research.google.com/github/Niranjana-biju/Intellectual-Navigator/blob/main/Intellectual_Navigator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wikipedia-api sentence-transformers networkx numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 5.4 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import networkx as nx
import wikipediaapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

wiki = wikipediaapi.Wikipedia('MyProject/1.0', 'en')
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [3]:
TOPIC_POOL = [
    # Science
    "Quantum mechanics", "Theory of relativity", "Evolution",
    "Neuroscience", "Genetics", "Thermodynamics", "Astrophysics",
    "String theory", "Dark matter", "Black hole", "Particle physics",
    "Nuclear physics", "Electromagnetism", "Optics", "Biophysics",
    "Biochemistry", "Organic chemistry", "Periodic table", "Astronomy",
    "Cosmology", "Climate change", "Oceanography", "Geology",
    "Meteorology", "Ecology",

    # Technology
    "Machine learning", "Neural network", "Robotics",
    "Cryptography", "Blockchain", "Virtual reality",
    "Computer vision", "Natural language processing",
    "Quantum computing", "Internet of things", "Cybersecurity",
    "Cloud computing", "Augmented reality", "3D printing",
    "Nanotechnology", "Biotechnology", "Space exploration",
    "Renewable energy", "Nuclear energy", "Electric vehicle",

    # Philosophy
    "Stoicism", "Existentialism", "Ethics", "Epistemology",
    "Philosophy of mind", "Free will", "Consciousness",
    "Metaphysics", "Logic", "Nihilism", "Utilitarianism",
    "Moral philosophy", "Political philosophy", "Aesthetics",
    "Phenomenology", "Rationalism", "Empiricism", "Humanism",
    "Determinism", "Absurdism",

    # Psychology
    "Cognitive bias", "Behavioral economics", "Memory",
    "Sleep", "Motivation", "Emotion", "Personality psychology",
    "Social psychology", "Cognitive psychology", "Psychoanalysis",
    "Positive psychology", "Developmental psychology", "Trauma",
    "Anxiety", "Depression", "Addiction", "Mindfulness",
    "Decision making", "Perception", "Learning",

    # Mathematics
    "Game theory", "Chaos theory", "Statistics",
    "Probability", "Number theory", "Topology",
    "Linear algebra", "Calculus", "Graph theory",
    "Set theory", "Combinatorics", "Information theory",
    "Fractal", "Prime number",

    # History
    "Roman Empire", "World War II", "Renaissance",
    "Industrial Revolution", "Cold War", "Ancient Greece",
    "Ottoman Empire", "French Revolution", "World War I",
    "Byzantine Empire", "Mongol Empire", "Ancient Egypt",
    "Medieval history", "Age of Enlightenment", "Colonialism",
    "American Revolution", "Russian Revolution",
    "Alexander the Great", "Julius Caesar",

    # Economics
    "Capitalism", "Supply and demand", "Inflation",
    "Stock market", "Globalization", "Microeconomics",
    "Macroeconomics", "Cryptocurrency", "Marxism",
    "Keynesian economics", "Free market", "Trade",
    "Poverty", "Inequality", "Taxation",
    "Central bank", "Monetary policy", "Recession",

    # Biology
    "DNA", "Cell biology", "Ecology", "Immunology",
    "Microbiology", "Anatomy", "Photosynthesis",
    "Natural selection", "Biodiversity", "Virus",
    "Bacteria", "Human genome", "Stem cell",
    "Cancer", "Aging", "Symbiosis",

    # Society & Culture
    "Anthropology", "Sociology", "Linguistics",
    "Mythology", "Religion", "Democracy",
    "Feminism", "Nationalism", "Human rights",
    "Social media", "Propaganda", "Culture",
    "Identity", "Language", "Education",
    "Media", "Urbanization",

    # Arts
    "Jazz", "Classical music", "Cinema",
    "Literature", "Architecture", "Photography",
    "Painting", "Sculpture", "Theatre",
    "Poetry", "Novel", "Opera",
    "Hip hop", "Rock music", "Folk music",

    # Health & Medicine
    "Meditation", "Nutrition", "Exercise physiology",
    "Mental health", "Neurology", "Pharmacology",
    "Epidemiology", "Surgery", "Immunization",
    "Public health", "Yoga", "Cognitive behavioral therapy",
    "Placebo effect",

    # Physics concepts
    "Entropy", "Gravity", "Time", "Space",
    "Wave", "Energy", "Matter", "Light",
    "Magnetism", "Electricity",

    # Music
    "Blues music", "African music", "Music theory",
    "Rhythm", "Improvisation", "Musical instrument",
    "Soul music", "Gospel music", "Reggae",

    # Space
    "Spacetime", "Gravitational wave", "Neutron star",
    "Supernova", "Galaxy", "Universe", "Big Bang",
    "Solar system", "Quantum gravity", "Event horizon",
    "Wormhole",

    # Ancient philosophy
    "Ancient Greek philosophy", "Virtue ethics",
    "Epicureanism", "Cynicism", "Platonism",
    "Aristotle", "Marcus Aurelius", "Seneca", "Epictetus",

    # Culture & identity
    "African American history", "Civil rights movement",
    "Slavery", "Cultural movement", "Dance",
    "Rhythm and blues", "Spirituality", "Buddhism",
    "Hinduism", "Islam", "Christianity", "Atheism",
    "Folklore",
]

TOPIC_POOL = list(dict.fromkeys(TOPIC_POOL))
print(f"Topic pool ready — {len(TOPIC_POOL)} topics")

Topic pool ready — 248 topics


In [4]:
print("Pre-embedding all topics — takes about 30 seconds...")
topic_pool_embs = model.encode(TOPIC_POOL, batch_size=64, show_progress_bar=True)
print("All topics embedded successfully")

Pre-embedding all topics — takes about 30 seconds...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

All topics embedded successfully


In [5]:
def clean_text(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[.*?\]', '', text)
    return text

def get_embedding(text):
    return model.encode(text)

def fetch_summary(topic):
    page = wiki.page(topic)
    if not page.exists():
        return None
    if len(page.summary) < 100:
        return None
    full_summary = clean_text(page.summary)

    trimmed = full_summary[:500]
    last_stop = trimmed.rfind('.')
    if last_stop != -1:
        trimmed = trimmed[:last_stop + 1]
    return trimmed

def fetch_topic(topic):
    summary = fetch_summary(topic)
    if not summary:
        print(f"Could not fetch '{topic}' from Wikipedia.")
        return None, None    # no summary, no topic

    page = wiki.page(topic)

    topic_words = [w.lower() for w in topic.split()]

    abbr = "".join(w[0].lower() for w in topic.split() if w[0].isalpha())

    wiki_links = [
        link for link in list(page.links.keys())[:500]
        if not link.startswith(("Help:", "Wikipedia:", "Portal:",
                                "File:", "Template:", "Category:",
                                "Talk:", "User:"))
        and len(link) > 4
        and not link[0].isdigit()
        and "disambiguation" not in link.lower()
        and "list of" not in link.lower()
        and not any(w in link.lower() for w in topic_words)
        and abbr not in link.lower()
        and "automat" not in link.lower()
    ]

    combined_pool = list(dict.fromkeys(TOPIC_POOL + wiki_links))
    combined_embs = model.encode(
        combined_pool, batch_size=64, show_progress_bar=False
    )
    topic_emb = get_embedding(summary)
    sims = cosine_similarity([topic_emb], combined_embs)[0]

    candidates = []
    for i, t in enumerate(combined_pool):
        if t.lower() == topic.lower():
            continue
        if t in curiosity_trail:
            continue
        candidates.append((t, float(sims[i])))

    candidates.sort(key=lambda x: x[1], reverse=True)

    final = []
    seen_words = set()
    for t, s in candidates:
        words = set(t.lower().split())
        if not words & seen_words:
            final.append(t)
            seen_words.update(words)
        if len(final) >= 20:
            break

    return summary, final

print("Cell 5 ready")

Cell 5 ready


In [6]:
curiosity_trail = []

def get_curiosity_vector():
    if not curiosity_trail:
        return None
    embeddings = [get_embedding(t) for t in curiosity_trail]
    return np.mean(embeddings, axis=0)

def surprise_score(candidate, curiosity_vector):
    if candidate in curiosity_trail:
        return 0.0
    candidate_emb = get_embedding(candidate)
    similarity = cosine_similarity([candidate_emb], [curiosity_vector])[0][0]
    return float(similarity)

print("Curiosity trail ready")

Curiosity trail ready


In [7]:
def recommend(topic, top_n=5):
    print(f"\nFetching: {topic}...")
    summary, links = fetch_topic(topic)
    if not summary:
        return []

    curiosity_trail.append(topic)
    curiosity_vector = get_curiosity_vector()

    scores = {}
    for link in links:
        scores[link] = surprise_score(link, curiosity_vector)

    top_topics = sorted(scores, key=scores.get, reverse=True)[:top_n]

    print(f"\nSummary: {summary}")
    print(f"\nYour curiosity trail: {' → '.join(curiosity_trail)}")
    print(f"\nRecommended next topics:")
    for i, t in enumerate(top_topics, 1):
        print(f"   {i}. {t}  (score: {round(scores[t], 2)})")

    return top_topics

print("Recommend function ready")

Recommend function ready


In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_curiosity_trail():
    if len(curiosity_trail) < 2:
        print("Visit at least 2 topics first.")
        return

    G = nx.DiGraph()
    for i in range(len(curiosity_trail) - 1):
        G.add_edge(curiosity_trail[i], curiosity_trail[i + 1])

    # Left to right layout
    pos = {topic: (i * 4, 0) for i, topic in enumerate(curiosity_trail)}

    n = len(curiosity_trail)
    plt.figure(figsize=(max(14, n * 4.5), 6))
    plt.title("Your Curiosity Trail",
              fontsize=18, fontweight='bold',
              color='#1E1B4B', pad=20)

    # Edges
    nx.draw_networkx_edges(G, pos,
                           edge_color='#4F46E5',
                           arrows=True,
                           arrowsize=25,
                           width=2.5,
                           alpha=0.8,
                           min_source_margin=60,
                           min_target_margin=60)

    # Node colours
    node_colors = []
    for topic in curiosity_trail:
        if topic == curiosity_trail[0]:
            node_colors.append('#4F46E5')
        elif topic == curiosity_trail[-1]:
            node_colors.append('#059669')
        else:
            node_colors.append('#7C3AED')

    nx.draw_networkx_nodes(G, pos,
                           nodelist=curiosity_trail,
                           node_color=node_colors,
                           node_size=5000,
                           alpha=0.95)

    # Shorten long labels to fit inside nodes
    short_labels = {
        topic: (topic[:12] + '..') if len(topic) > 12 else topic
        for topic in curiosity_trail
    }
    nx.draw_networkx_labels(G, pos,
                            labels=short_labels,
                            font_size=8,
                            font_weight='bold',
                            font_color='white')

    # Full topic names as step labels above nodes
    step_pos = {topic: (i * 4, 0.7) for i, topic in enumerate(curiosity_trail)}
    step_labels = {topic: f"Step {i+1}\n{topic}" for i, topic in enumerate(curiosity_trail)}
    nx.draw_networkx_labels(G, step_pos,
                            labels=step_labels,
                            font_size=8,
                            font_color='#374151')

    # Trail text at bottom
    trail_text = " → ".join(curiosity_trail)
    plt.figtext(0.5, 0.02, trail_text,
                ha='center', fontsize=9,
                color='#6B7280', style='italic')

    # Legend
    start   = mpatches.Patch(color='#4F46E5', label='Start')
    middle  = mpatches.Patch(color='#7C3AED', label='Visited')
    current = mpatches.Patch(color='#059669', label='Current')
    plt.legend(handles=[start, middle, current],
               loc='upper left', fontsize=10)

    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("draw_curiosity_trail ready")

draw_curiosity_trail ready


In [9]:
curiosity_trail = []
recommendations = recommend("Artificial Intelligence")


Fetching: Artificial Intelligence...

Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.

Your curiosity trail: Artificial Intelligence

Recommended next topics:
   1. Machine learning  (score: 0.7)
   2. Autonomous agent  (score: 0.63)
   3. Neural network  (score: 0.59)
   4. Computing  (score: 0.58)
   5. Cognitive architecture  (score: 0.54)


In [10]:
recommendations = recommend("Neuroscience")


Fetching: Neuroscience...

Summary: Neuroscience is the scientific study of the nervous system (the brain, spinal cord, and peripheral nervous system), its functions, and its disorders. It is a multidisciplinary science that combines physiology, anatomy, molecular biology, developmental biology, cytology, psychology, physics, computer science, chemistry, medicine, statistics, and mathematical modeling to understand the fundamental and emergent properties of neurons, glia, and neural circuits.

Your curiosity trail: Artificial Intelligence → Neuroscience

Recommended next topics:
   1. Cerebral cortex  (score: 0.65)
   2. Neurology  (score: 0.59)
   3. Biology  (score: 0.55)
   4. Hippocampus  (score: 0.53)
   5. Anatomy  (score: 0.51)


In [11]:
recommendations = recommend("Consciousness")


Fetching: Consciousness...

Summary: Consciousness is being aware of something internal to one's self, or of states or objects in one's external environment. It has been the topic of extensive explanations, analyses, and debate among philosophers, scientists, and theologians for millennia. There is no consensus on what exactly needs to be studied, or whether consciousness can be considered a scientific concept. In some explanations it is synonymous with mind, while in others it is considered an aspect of it.

Your curiosity trail: Artificial Intelligence → Neuroscience → Consciousness

Recommended next topics:
   1. Human brain  (score: 0.75)
   2. Perception  (score: 0.63)
   3. Neurophysiology  (score: 0.59)
   4. Cognitive psychology  (score: 0.59)
   5. Neurology  (score: 0.58)


In [12]:
recommendations = recommend("Meditation")


Fetching: Meditation...

Summary: Meditation is a practice in which an individual uses a technique or combination of techniques to train attention and awareness and detach from "discursive, ruminating thought", achieving a mentally clear and emotionally calm and stable state, while not intending to analyze the effects, to judge its outcomes, or to create any expectation with regard to the process. Meditation techniques are broadly classified into focused (or concentrative) and open monitoring methods.

Your curiosity trail: Artificial Intelligence → Neuroscience → Consciousness → Meditation

Recommended next topics:
   1. Philosophy of mind  (score: 0.6)
   2. Mindfulness  (score: 0.58)
   3. Conscious breathing  (score: 0.57)
   4. Awareness  (score: 0.54)
   5. Spirituality  (score: 0.54)


In [13]:
recommendations = recommend("Spirituality")


Fetching: Spirituality...

Summary: The meaning of spirituality has developed and expanded over time, and various meanings can be found alongside each other. Traditionally, spirituality referred to a religious process of re-formation which "aims to recover the original shape of man", oriented at "the image of God" as exemplified by the founders and sacred texts of the religions of the world.

Your curiosity trail: Artificial Intelligence → Neuroscience → Consciousness → Meditation → Spirituality

Recommended next topics:
   1. Mindfulness  (score: 0.63)
   2. Buddhism  (score: 0.6)
   3. Belief  (score: 0.57)
   4. Afterlife  (score: 0.53)
   5. Buddha  (score: 0.53)


In [14]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def evaluate(test_topics, top_n=5):
    print("EVALUATION RESULTS")
    print("-" * 60)

    relevance_scores = []
    response_times   = []

    for topic in test_topics:
        global curiosity_trail
        curiosity_trail = []

        start = time.time()
        summary, links = fetch_topic(topic)
        if not summary:
            print(f"{topic:25} → FAILED")
            continue

        curiosity_trail.append(topic)
        curiosity_vector = get_curiosity_vector()

        scores    = {link: surprise_score(link, curiosity_vector) for link in links}
        top5      = sorted(scores, key=scores.get, reverse=True)[:top_n]
        avg_score = round(np.mean([scores[t] for t in top5]), 2)
        time_taken = round(time.time() - start, 1)

        relevance_scores.append(avg_score)
        response_times.append(time_taken)

        print(f"{topic:25} | Score: {avg_score} | Time: {time_taken}s")

    print("-" * 40)
    print(f"Average Relevance Score : {round(np.mean(relevance_scores), 2)}")
    print(f"Average Response Time   : {round(np.mean(response_times), 1)} seconds")
    print("-" * 40)

print("evaluate ready")

evaluate ready


In [15]:
test_topics = [
    "Artificial Intelligence",
    "Stoicism",
    "Black hole",
    "Jazz",
    "Neuroscience",
    "Democracy",
    "Evolution",
    "Meditation",
    "World War II",
    "Quantum mechanics"
]

evaluate(test_topics)

EVALUATION RESULTS
------------------------------------------------------------
Artificial Intelligence   | Score: 0.61 | Time: 7.4s
Stoicism                  | Score: 0.46 | Time: 3.9s
Black hole                | Score: 0.6 | Time: 8.4s
Jazz                      | Score: 0.54 | Time: 7.8s
Neuroscience              | Score: 0.63 | Time: 6.2s
Democracy                 | Score: 0.63 | Time: 3.6s
Evolution                 | Score: 0.61 | Time: 2.1s
Meditation                | Score: 0.6 | Time: 3.1s
World War II              | Score: 0.6 | Time: 5.1s
Quantum mechanics         | Score: 0.58 | Time: 3.7s
----------------------------------------
Average Relevance Score : 0.59
Average Response Time   : 5.1 seconds
----------------------------------------


In [16]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.7 MB/s eta 0:00:00


In [17]:
app_code = '''
import streamlit as st
import re
import numpy as np
import wikipediaapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(
    page_title="Intellectual Navigator",
    page_icon="🐇",
    layout="centered"
)

st.markdown("""
<style>
    .main { background-color: #0f0f1a; }
    .block-container { padding-top: 2rem; }
    h1 { color: #e8e8f0 !important; font-size: 2.2rem !important; }
    .subtitle { color: #6b6b8a; font-size: 0.95rem; margin-bottom: 2rem; }
    .summary-box { background: #1a1a2e; border-left: 3px solid #7c6af7;
                   border-radius: 6px; padding: 1rem 1.2rem;
                   color: #c8c8e0; font-style: italic;
                   font-size: 0.95rem; line-height: 1.7;
                   margin-bottom: 1.5rem; }
    .trail-box { background: #12122a; border: 1px solid #2a2a4a;
                 border-radius: 6px; padding: 0.8rem 1rem;
                 color: #a89cf7; font-size: 0.85rem;
                 margin-bottom: 1.5rem; }
    .rec-title { color: #a89cf7; font-size: 1rem;
                 font-weight: 700; margin-bottom: 0.2rem; }
    .rec-score { color: #4a4a6a; font-size: 0.78rem; }
    .depth-badge { background: #2a1a4a; color: #a89cf7;
                   padding: 0.2rem 0.7rem; border-radius: 20px;
                   font-size: 0.8rem; display: inline-block;
                   margin-bottom: 1rem; }
    .section-label { color: #4a4a6a; font-size: 0.75rem;
                     letter-spacing: 0.15em; text-transform: uppercase;
                     margin-bottom: 0.5rem; }
    div[data-testid="stButton"] button {
        background: #7c6af7 !important; color: white !important;
        border: none !important; border-radius: 6px !important;
        font-size: 0.85rem !important; width: 100% !important;
    }
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource
def load_wiki():
    return wikipediaapi.Wikipedia("MyProject/1.0", "en")

model = load_model()
wiki  = load_wiki()

TOPIC_POOL = [
    "Quantum mechanics", "Theory of relativity", "Evolution",
    "Neuroscience", "Genetics", "Thermodynamics", "Astrophysics",
    "String theory", "Dark matter", "Black hole", "Particle physics",
    "Nuclear physics", "Electromagnetism", "Optics", "Biophysics",
    "Biochemistry", "Organic chemistry", "Periodic table", "Astronomy",
    "Cosmology", "Climate change", "Oceanography", "Geology",
    "Meteorology", "Ecology", "Machine learning", "Neural network",
    "Robotics", "Cryptography", "Blockchain", "Virtual reality",
    "Computer vision", "Natural language processing", "Quantum computing",
    "Internet of things", "Cybersecurity", "Cloud computing",
    "Augmented reality", "3D printing", "Nanotechnology", "Biotechnology",
    "Space exploration", "Renewable energy", "Nuclear energy",
    "Electric vehicle", "Stoicism", "Existentialism", "Ethics",
    "Epistemology", "Philosophy of mind", "Free will", "Consciousness",
    "Metaphysics", "Logic", "Nihilism", "Utilitarianism",
    "Moral philosophy", "Political philosophy", "Aesthetics",
    "Phenomenology", "Rationalism", "Empiricism", "Humanism",
    "Determinism", "Absurdism", "Cognitive bias", "Behavioral economics",
    "Memory", "Sleep", "Motivation", "Emotion", "Personality psychology",
    "Social psychology", "Cognitive psychology", "Psychoanalysis",
    "Positive psychology", "Developmental psychology", "Trauma",
    "Anxiety", "Depression", "Addiction", "Mindfulness",
    "Decision making", "Perception", "Learning", "Game theory",
    "Chaos theory", "Statistics", "Probability", "Number theory",
    "Topology", "Linear algebra", "Calculus", "Graph theory",
    "Set theory", "Combinatorics", "Information theory", "Fractal",
    "Prime number", "Roman Empire", "World War II", "Renaissance",
    "Industrial Revolution", "Cold War", "Ancient Greece",
    "Ottoman Empire", "French Revolution", "World War I",
    "Byzantine Empire", "Mongol Empire", "Ancient Egypt",
    "Medieval history", "Age of Enlightenment", "Colonialism",
    "American Revolution", "Russian Revolution", "Alexander the Great",
    "Julius Caesar", "Capitalism", "Supply and demand", "Inflation",
    "Stock market", "Globalization", "Microeconomics", "Macroeconomics",
    "Cryptocurrency", "Marxism", "Keynesian economics", "Free market",
    "Trade", "Poverty", "Inequality", "Taxation", "Central bank",
    "Monetary policy", "Recession", "DNA", "Cell biology", "Immunology",
    "Microbiology", "Anatomy", "Photosynthesis", "Natural selection",
    "Biodiversity", "Virus", "Bacteria", "Human genome", "Stem cell",
    "Cancer", "Aging", "Symbiosis", "Anthropology", "Sociology",
    "Linguistics", "Mythology", "Religion", "Democracy", "Feminism",
    "Nationalism", "Human rights", "Social media", "Propaganda",
    "Culture", "Identity", "Language", "Education", "Media",
    "Urbanization", "Jazz", "Classical music", "Cinema", "Literature",
    "Architecture", "Photography", "Painting", "Sculpture", "Theatre",
    "Poetry", "Novel", "Opera", "Hip hop", "Rock music", "Folk music",
    "Meditation", "Nutrition", "Exercise physiology", "Mental health",
    "Neurology", "Pharmacology", "Epidemiology", "Surgery",
    "Immunization", "Public health", "Yoga",
    "Cognitive behavioral therapy", "Placebo effect", "Entropy",
    "Gravity", "Time", "Space", "Wave", "Energy", "Matter", "Light",
    "Magnetism", "Electricity", "Blues music", "Music theory",
    "Rhythm", "Improvisation", "Spacetime", "Gravitational wave",
    "Neutron star", "Supernova", "Galaxy", "Universe", "Big Bang",
    "Solar system", "Event horizon", "Wormhole",
    "Ancient Greek philosophy", "Virtue ethics", "Epicureanism",
    "Platonism", "Aristotle", "Marcus Aurelius", "Seneca", "Epictetus",
    "African American history", "Civil rights movement", "Slavery",
    "Dance", "Spirituality", "Buddhism", "Hinduism", "Islam",
    "Christianity", "Atheism", "Folklore", "Computer science",
    "Software engineering", "Data science", "Algorithms",
    "Operating system", "Database", "Programming language",
]

TOPIC_POOL = list(dict.fromkeys(TOPIC_POOL))

@st.cache_data
def get_pool_embeddings():
    return model.encode(TOPIC_POOL, batch_size=64, show_progress_bar=False)

pool_embs = get_pool_embeddings()

def clean_text(text):
    text = re.sub(r"\s+", " ", text.strip())
    text = re.sub(r"\[.*?\]", "", text)
    return text

def fetch_summary(topic):
    page = wiki.page(topic)
    if not page.exists() or len(page.summary) < 100:
        return None
    full    = clean_text(page.summary)
    trimmed = full[:500]
    last    = trimmed.rfind(".")
    return trimmed[:last + 1] if last != -1 else trimmed

def fetch_candidates(topic, trail):
    page        = wiki.page(topic)
    topic_words = [w.lower() for w in topic.split()]
    abbr        = "".join(w[0].lower() for w in topic.split() if w[0].isalpha())

    wiki_links = [
        link for link in list(page.links.keys())[:500]
        if not link.startswith(("Help:", "Wikipedia:", "Portal:",
                                "File:", "Template:", "Category:",
                                "Talk:", "User:"))
        and len(link) > 4
        and not link[0].isdigit()
        and "disambiguation" not in link.lower()
        and "list of" not in link.lower()
        and not any(w in link.lower() for w in topic_words)
        and abbr not in link.lower()
        and "automat" not in link.lower()
    ]

    combined  = list(dict.fromkeys(TOPIC_POOL + wiki_links))
    comb_embs = model.encode(combined, batch_size=64, show_progress_bar=False)
    topic_emb = model.encode(fetch_summary(topic) or topic)
    sims      = cosine_similarity([topic_emb], comb_embs)[0]

    candidates = [
        (t, float(sims[i])) for i, t in enumerate(combined)
        if t.lower() != topic.lower() and t not in trail
    ]
    candidates.sort(key=lambda x: x[1], reverse=True)

    final, seen = [], set()
    for t, s in candidates:
        words = set(t.lower().split())
        if not words & seen:
            final.append((t, s))
            seen.update(words)
        if len(final) >= 20:
            break
    return final[:20]

def get_curiosity_vector(trail):
    return np.mean(model.encode(trail), axis=0)

def score_candidates(candidates, trail):
    if not trail:
        return {t: s for t, s in candidates[:5]}
    curv   = get_curiosity_vector(trail)
    scores = {}
    for t, _ in candidates:
        emb       = model.encode(t)
        sim       = cosine_similarity([emb], [curv])[0][0]
        scores[t] = float(sim) if t not in trail else 0.0
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5])

if "trail"   not in st.session_state: st.session_state.trail   = []
if "current" not in st.session_state: st.session_state.current = None
if "summary" not in st.session_state: st.session_state.summary = None
if "recs"    not in st.session_state: st.session_state.recs    = {}

def explore(topic):
    summary = fetch_summary(topic)
    if not summary:
        st.error(f"Could not find \'{topic}\' on Wikipedia.")
        return
    candidates = fetch_candidates(topic, st.session_state.trail)
    recs       = score_candidates(candidates, st.session_state.trail)
    st.session_state.trail.append(topic)
    st.session_state.current = topic
    st.session_state.summary = summary
    st.session_state.recs    = recs

st.markdown("<h1>🐇 Intellectual Navigator</h1>", unsafe_allow_html=True)
st.markdown("<p class=\\"subtitle\\">Enter any topic. Follow the thread. See where curiosity leads.</p>", unsafe_allow_html=True)

col1, col2 = st.columns([4, 1])
with col1:
    topic_input = st.text_input("", placeholder="Try: Stoicism, Black hole, Jazz...", label_visibility="collapsed")
with col2:
    search_btn = st.button("Explore →")

if search_btn and topic_input.strip():
    st.session_state.trail = []
    explore(topic_input.strip())

if st.session_state.trail:
    trail_text = " → ".join(st.session_state.trail)
    depth      = len(st.session_state.trail)
    st.markdown(f"<div class=\\"depth-badge\\">Depth: {depth}</div>", unsafe_allow_html=True)
    st.markdown(f"<div class=\\"trail-box\\">🧭 {trail_text}</div>", unsafe_allow_html=True)

if st.session_state.summary:
    st.markdown("<div class=\\"section-label\\">About this topic</div>", unsafe_allow_html=True)
    st.markdown(f"<div class=\\"summary-box\\">{st.session_state.summary}</div>", unsafe_allow_html=True)

if st.session_state.recs:
    st.markdown("<div class=\\"section-label\\">Where to go next</div>", unsafe_allow_html=True)
    for topic, score in st.session_state.recs.items():
        col1, col2 = st.columns([5, 1])
        with col1:
            st.markdown(f"<div class=\\"rec-title\\">{topic}</div><div class=\\"rec-score\\">Score: {round(score, 2)}</div>", unsafe_allow_html=True)
        with col2:
            if st.button("→", key=f"btn_{topic}"):
                explore(topic)

if st.session_state.trail:
    st.markdown("---")
    if st.button("↩ Start Over"):
        st.session_state.trail   = []
        st.session_state.current = None
        st.session_state.summary = None
        st.session_state.recs    = {}
        st.rerun()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully


<>:133: SyntaxWarning: invalid escape sequence '\s'
<>:133: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2287/1499362175.py:133: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(r"\s+", " ", text.strip())


In [21]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess
import time

ngrok.kill()

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")

if not ngrok_token:
    raise ValueError("NGROK_AUTH_TOKEN not found in Colab Secrets")

ngrok.set_auth_token(ngrok_token)


subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

time.sleep(5)


public_url = ngrok.connect(8501)

print("Open this link in your browser:")
print(public_url)

Open this link in your browser:
NgrokTunnel: "https://merocrine-gratifyingly-ozella.ngrok-free.dev" -> "http://localhost:8501"
